In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing
import pandas as pd


In [2]:
data = fetch_california_housing(as_frame=True)
df = pd.concat(  [data.data, data.target.rename("HousePrice")],  axis=1)
print(df.head())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  HousePrice  
0    -122.23       4.526  
1    -122.22       3.585  
2    -122.24       3.521  
3    -122.25       3.413  
4    -122.25       3.422  


In [3]:
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]
print(X.shape)
print(y.shape)

(20640, 8)
(20640,)


In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled[:5])

[[ 2.34476576  0.98214266  0.62855945 -0.15375759 -0.9744286  -0.04959654
   1.05254828 -1.32783522]
 [ 2.33223796 -0.60701891  0.32704136 -0.26333577  0.86143887 -0.09251223
   1.04318455 -1.32284391]
 [ 1.7826994   1.85618152  1.15562047 -0.04901636 -0.82077735 -0.02584253
   1.03850269 -1.33282653]
 [ 0.93296751  1.85618152  0.15696608 -0.04983292 -0.76602806 -0.0503293
   1.03850269 -1.33781784]
 [-0.012881    1.85618152  0.3447108  -0.03290586 -0.75984669 -0.08561576
   1.03850269 -1.33781784]]


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)

(16512, 8)
(4128, 8)


In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)
print(lr_rmse)
print(lr_r2)

0.7455813830127763
0.575787706032451


In [7]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)
print("RMSE =", ridge_rmse)
print("R2 =", ridge_r2)

RMSE = 0.7455542909384612
R2 = 0.5758185345441319


In [8]:
tree = DecisionTreeRegressor( random_state=42)
tree.fit(X_train, y_train)
train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)
train_rmse = np.sqrt( mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(   mean_squared_error(y_test, test_pred))
print("Train RMSE =", train_rmse)
print("Test RMSE =", test_rmse)

Train RMSE = 3.0176010694886063e-16
Test RMSE = 0.7021725586639983


In [9]:
cv_scores = cross_val_score( tree, X_scaled, y, cv=5, scoring="neg_root_mean_squared_error")
cv_rmse = -cv_scores.mean()
print("Cross Validation RMSE =", cv_rmse)

Cross Validation RMSE = 0.8961572070066897


In [10]:
param_grid = { "max_depth": [3, 5, 7, 10], "min_samples_split": [2, 5, 10]}
grid = GridSearchCV( DecisionTreeRegressor(random_state=42 ), param_grid, cv=5,scoring="neg_root_mean_squared_error")
grid.fit(X_train, y_train)
print(grid.best_params_)

{'max_depth': 10, 'min_samples_split': 10}


In [11]:
best_tree = grid.best_estimator_
best_pred = best_tree.predict(X_test)
best_rmse = np.sqrt( mean_squared_error(y_test,best_pred))
best_r2 = r2_score(  y_test,  best_pred)
print("RMSE =", best_rmse)
print("R2 =", best_r2)

RMSE = 0.6454300828015771
R2 = 0.6820992539714815


In [12]:
results = pd.DataFrame({
"Model": ["Linear Regression","Ridge Regression","Tuned Decision Tree" ],
"RMSE": [lr_rmse,ridge_rmse,best_rmse ],
    "R2 Score": [lr_r2,ridge_r2,best_r2 ]})

print(results)

                 Model      RMSE  R2 Score
0    Linear Regression  0.745581  0.575788
1     Ridge Regression  0.745554  0.575819
2  Tuned Decision Tree  0.645430  0.682099
